# M2.4 · Decision tree

Gradyansız öğrenen model. Altı örnekte Gini ve en iyi bölünmeyi elle buluyor, saf Python'da özyinelemeli bir ağaç kuruyor, NumPy'da tüm özelliklerin tüm eşiklerini cumsum ile tek seferde değerlendiriyor, PyTorch'ta türevlenebilir "yumuşak" bir karar düğümünü autograd ile eğitiyor ve en sonda örnek ağırlıklı bölünmeyi (boosting'in ihtiyacı) döngüsüz hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-4-tree.html](https://m-gonul.github.io/pytorch-lab/m2-4-tree.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_4_tree.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref_best(X, y):\n    n, d = len(X), len(X[0])\n    G = lambda s: 1 - sum((s.count(c) / len(s)) ** 2 for c in set(s)) if s else 0.0\n    best = None\n    for j in range(d):\n        vals = sorted(set(r[j] for r in X))\n        for a, b in zip(vals, vals[1:]):\n            t = (a + b) / 2\n            L = [y[i] for i in range(n) if X[i][j] <= t]; R_ = [y[i] for i in range(n) if X[i][j] > t]\n            s = len(L) / n * G(L) + len(R_) / n * G(R_)\n            if best is None or s < best[2] - 1e-12:\n                best = (j, t, s)\n    return best\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"sklearn\"}))\n    R.test(\"gini\", lambda: close([need(ns, \"gini\")([0, 0, 1, 1]), need(ns, \"gini\")([1, 1, 1, 0]), need(ns, \"gini\")([2, 2])], [0.5, 0.375, 0.0]))\n    X1, y1 = [[1], [2], [3], [4], [5], [6]], [0, 0, 1, 1, 1, 0]\n\n    def hand():\n        j, t, s = need(ns, \"best_split\")(X1, y1)\n        ensure(j == 0, \"özellik 0\"); close([t, s], [2.5, 0.25])\n        ensure(need(ns, \"best_split\")([[1], [1]], [0, 1]) is None, \"tek farklı değer → None\")\n    R.test(\"best_split: elle örnek (2.5, 0.25), bölünemezse None\", hand)\n    rng = random.Random(0)\n    X = [[round(rng.uniform(0, 10), 3) for _ in range(3)] for _ in range(40)]\n    y = [int(x[1] > 4) ^ int(x[2] > 7) for x in X]\n\n    def rnd():\n        got, want = need(ns, \"best_split\")(X, y), ref_best(X, y)\n        ensure(got[0] == want[0], f\"özellik {got[0]}, {want[0]} bekleniyordu\"); close(got[1:], want[1:])\n    R.test(\"rastgele 3 özellik: referansla aynı bölünme\", rnd)\n\n    def tree():\n        T = need(ns, \"build_tree\")(X, y, 4)\n        acc = sum(need(ns, \"predict_tree\")(T, x) == t for x, t in zip(X, y)) / 40\n        ensure(acc == 1.0, f\"derinlik 4 ile eğitim doğruluğu {acc}, 1.0 bekleniyordu (XOR yapısı)\")\n        T1 = need(ns, \"build_tree\")(X, y, 1)\n        ensure(\"leaf\" in T1[\"left\"] and \"leaf\" in T1[\"right\"], \"max_depth = 1 tek bölünme olmalı\")\n        T0 = need(ns, \"build_tree\")(X1, y1, 0)\n        ensure(T0 == {\"leaf\": 0}, \"max_depth = 0: çoğunluk yaprağı (eşitlikte küçük etiket)\")\n        return \"derinlik 4 ağaç XOR benzeri sınırı tam ayırıyor\"\n    R.test(\"build_tree + predict_tree\", tree)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef brute(X, y, C):\n    n, d = X.shape\n    best = None\n    for j in range(d):\n        v = np.unique(X[:, j])\n        for t in (v[:-1] + v[1:]) / 2:\n            m = X[:, j] <= t\n            def G(s):\n                p = np.bincount(s, minlength=C) / len(s)\n                return 1 - (p ** 2).sum()\n            s = m.mean() * G(y[m]) + (1 - m.mean()) * G(y[~m])\n            if best is None or s < best[2] - 1e-12:\n                best = (j, t, s)\n    return best\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"torch\", \"sklearn\"}))\n    f = lambda *a: need(ns, \"best_split_np\")(*a)\n    R.test(\"elle örnek (0, 2.5, 0.25)\", lambda: close(f(np.array([[1.0], [2], [3], [4], [5], [6]]), np.array([0, 0, 1, 1, 1, 0]), 2), [0, 2.5, 0.25]))\n    R.test(\"bölünemezse None\", lambda: ensure(f(np.array([[1.0], [1.0]]), np.array([0, 1]), 2) is None, \"None bekleniyordu\"))\n    rng = np.random.default_rng(0)\n\n    def rnd():\n        for C, d in [(2, 3), (4, 5)]:\n            X = rng.normal(size=(60, d)).round(2)\n            y = rng.integers(0, C, 60)\n            got, want = f(X, y, C), brute(X, y, C)\n            ensure(got[0] == want[0], f\"C={C}: özellik {got[0]}, {want[0]} bekleniyordu\")\n            close(got[1:], want[1:], what=f\"C={C}\")\n    R.test(\"rastgele veri (tekrarlı değerler dahil): kaba kuvvetle aynı\", rnd)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    X = torch.rand(400, 5, generator=g)\n    y = (X[:, 3] > 0.35).long()\n\n    def shape():\n        torch.manual_seed(0)\n        m = need(ns, \"SoftStump\", \"sınıf\")(5, 3)\n        ensure(isinstance(m, nn.Module), \"nn.Module olmalı\")\n        out = m(X)\n        ensure(tuple(out.shape) == (400, 3), f\"çıktı shape {tuple(out.shape)}\")\n        ensure(sum(p.numel() for p in m.parameters()) == 5 + 1 + 3 + 3, \"parametreler: kapı (d + 1) + iki yaprak (C + C)\")\n    R.test(\"SoftStump: shape ve parametre sayısı\", shape)\n\n    def tr():\n        torch.manual_seed(0)\n        m = need(ns, \"train_soft_stump\")(X, y, 2)\n        acc = (m(X).argmax(1) == y).float().mean().item()\n        ensure(acc >= 0.95, f\"doğruluk {acc:.2f}\")\n        lin = [mod for mod in m.modules() if isinstance(mod, nn.Linear)]\n        ensure(lin, \"kapı bir nn.Linear olmalı\")\n        w = lin[0].weight.detach().view(-1).abs()\n        ensure(int(w.argmax()) == 3, f\"kapı ağırlıkları {w.tolist()}; özellik 3'e yoğunlaşmalı\")\n        b = float(lin[0].bias.detach()); w3 = float(lin[0].weight.detach().view(-1)[3])\n        return f\"doğruluk {acc:.3f}; öğrenilen eşik ≈ {-b / w3:.3f} (gerçek 0.35)\"\n    R.test(\"eğitim: doğruluk ≥ 0.95, kapı doğru özelliği seçiyor\", tr)\n",
 "e": "import random\nimport torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef py_best(X, y):\n    n, d = len(X), len(X[0])\n    def G(s):\n        if not s:\n            return 0.0\n        c = {}\n        for v in s:\n            c[v] = c.get(v, 0) + 1\n        return 1 - sum((k / len(s)) ** 2 for k in c.values())\n    best = None\n    for j in range(d):\n        vals = sorted(set(r[j] for r in X))\n        for a, b in zip(vals, vals[1:]):\n            t = (a + b) / 2\n            L = [y[i] for i in range(n) if X[i][j] <= t]; Rr = [y[i] for i in range(n) if X[i][j] > t]\n            s = len(L) / n * G(L) + len(Rr) / n * G(Rr)\n            if best is None or s < best[2]:\n                best = (j, t, s)\n    return best\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    f = lambda *a: need(ns, \"best_split_torch\")(*a)\n    g = torch.Generator().manual_seed(0)\n    X = torch.randn(80, 4, generator=g, dtype=torch.float64).round(decimals=2)\n    y = torch.randint(0, 3, (80,), generator=g)\n\n    def unw():\n        got, want = f(X, y, 3), py_best(X.tolist(), y.tolist())\n        ensure(got[0] == want[0], f\"özellik {got[0]}, {want[0]} bekleniyordu\"); close(got[1:], want[1:])\n    R.test(\"ağırlıksız: saf Python referansla aynı\", unw)\n\n    def wt():\n        w = torch.randint(1, 4, (80,), generator=g)\n        got = f(X, y, 3, w.double())\n        Xd, yd = X.repeat_interleave(w, 0), y.repeat_interleave(w)\n        want = py_best(Xd.tolist(), yd.tolist())\n        ensure(got[0] == want[0], f\"özellik {got[0]}, {want[0]} bekleniyordu\"); close(got[1:], want[1:])\n        return \"ağırlık k = satırı k kez kopyalamak\"\n    R.test(\"tamsayı ağırlık = satır kopyalama\", wt)\n\n    def hand():\n        r = f(torch.tensor([[1.0], [2], [3], [4], [5], [6]]), torch.tensor([0, 0, 1, 1, 1, 0]), 2, torch.tensor([1.0, 1, 1, 1, 1, 10]))\n        close(r[1], 5.5, what=\"son örneğe ağırlık 10 verince eşik\")\n    R.test(\"ağır örnek bölünmeyi kendi yanına çekiyor\", hand)\n    if \"best_split_torch\" not in ns:\n        return\n    rng = random.Random(0)\n    Xb = torch.randn(2000, 20, generator=g)\n    yb = torch.randint(0, 2, (2000,), generator=g)\n    Xl, yl = Xb[:300].tolist(), yb[:300].tolist()\n    t_py = bench(lambda: py_best(Xl, yl), repeat=1, number=1)\n    t_vec = bench(lambda: ns[\"best_split_torch\"](Xb[:300], yb[:300], 2), repeat=3)\n    t_big = bench(lambda: ns[\"best_split_torch\"](Xb, yb, 2), repeat=3)\n    R.table(\"En iyi bölünme araması (d = 20)\", [\"yöntem\", \"n\", \"süre\"],\n            [[\"saf Python (her eşikte yeniden say)\", \"300\", fmt_time(t_py)],\n             [\"senin best_split_torch\", \"300\", fmt_time(t_vec)],\n             [\"senin best_split_torch\", \"2000\", fmt_time(t_big)]],\n            note=f\"n = 300'de {t_py / t_vec:,.0f}× hızlı. Saf Python O(n²·d); sıralama + cumsum O(n log n · d).\")\n"
}

In [ ]:
import importlib, labcheck, checks_m2_4_tree as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir karar ağacı, girdi uzayını eksenlere paralel dikdörtgenlere böler ve her bölgede sabit bir tahmin verir. Her iç düğüm bir soru sorar: $x_j\le t$ mi?

Bir bölünmenin iyiliği, çocuklarının **safsızlığıyla** ölçülür. Sınıf oranları $p_c$ için:

$$
\text{Gini: }G=1-\sum_c p_c^2,\qquad \text{Entropi: }H=-\sum_c p_c\log_2p_c,
\qquad
\text{skor}(j,t)=\frac{n_L}{n}G_L+\frac{n_R}{n}G_R .
$$

**CART** algoritması açgözlüdür: her düğümde tüm $(j,t)$ çiftleri arasından skoru en küçük olanı seçer, sonra iki çocukta aynı şeyi özyinelemeli yapar. Aday eşikler, bir özelliğin sıralı farklı değerlerinin orta noktalarıdır.

Ağaçta gradyan yok. Ama sert soru $[x_j\le t]$'yi yumuşak bir kapıyla, $\sigma\big((w^\top x+b)/\tau\big)$, değiştirince model türevlenebilir hâle gelir ve autograd ile eğitilebilir ("soft decision tree"). d aşaması bu köprüyü kuruyor.

## 1 · Elle

Altı örneği bir sayı doğrusuna yerleştir, renklerini işaretle. Beş aday eşiğin hepsini dene ve en iyisini bul.

<!-- çözüm -->

$t=2.5$ solu tamamen saflaştırıyor; sağda tek bir "yanlış" örnek kalıyor. $t=4.5$ ise hiç işe yaramıyor (ağırlıklı Gini kökle aynı). Ağacı büyütmeye devam etseydin sağ çocukta bir sonraki bölünme $t=5.5$ olurdu ve tüm yapraklar saflaşırdı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$x=[1,2,3,4,5,6]$, $y=[0,0,1,1,1,0]$. Gini $G=1-\sum_c p_c^2$**

- `groot`: Kök Gini
- `eroot`: Kök entropi (bit)

**Eşik $t=2.5$: sol $x\le t$, sağ $x>t$**

- `gl`: Sol Gini
- `gr`: Sağ Gini
- `wg`: Ağırlıklı Gini $\frac{n_L}{n}G_L+\frac{n_R}{n}G_R$

**Tüm eşikler (1.5, 2.5, 3.5, 4.5, 5.5) içinde**

- `w45`: $t=4.5$ ağırlıklı Gini
- `best`: En iyi eşik
- `gain`: Kazanç (kök − ağırlıklı)

In [ ]:
cevaplar = dict(
    groot=None,
    eroot=None,
    gl=None,
    gr=None,
    wg=None,
    w45=None,
    best=None,
    gain=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"groot":[0.5,0.001],"eroot":[1,0.001],"gl":[0,0.001],"gr":[0.375,0.001],"wg":[0.25,0.001],"w45":[0.5,0.002],"best":[2.5,0.001],"gain":[0.25,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. `X` liste listesi, `y` tamsayı etiketler.

- `gini(labels)`
- `best_split(X, y)` → `(j, t, skor)` ya da bölünecek eşik yoksa `None`. Aday eşikler her özellikte sıralı **farklı** değerlerin orta noktaları. Eşitlikte küçük özellik indeksi, sonra küçük eşik kazanır.
- `build_tree(X, y, max_depth)` — düğüm: yaprak `{"leaf": sınıf}` ya da `{"j": …, "t": …, "left": …, "right": …}`. Saf düğümde, `max_depth == 0`'da ya da bölünme yoksa yaprak; yaprak sınıfı çoğunluk (eşitlikte küçük etiket). Sol: $x_j\le t$.
- `predict_tree(tree, x)`

In [ ]:
%%writefile stage_b.py
def gini(labels):
    ...


def best_split(X, y):
    """→ (j, t, skor) ya da None"""
    ...


def build_tree(X, y, max_depth):
    ...


def predict_tree(tree, x):
    ...


X = [[1], [2], [3], [4], [5], [6]]
y = [0, 0, 1, 1, 1, 0]
print(best_split(X, y))             # (0, 2.5, 0.25)
tree = build_tree(X, y, 3)
print(tree)
print([predict_tree(tree, x) for x in X])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Tüm $(j,t)$ adaylarını **döngüsüz** değerlendir:

1. `order = np.argsort(X, axis=0, kind="stable")` — her sütunu ayrı sırala; `Xs = np.take_along_axis(X, order, 0)`.
2. Sıralı etiketlerin one-hot'ı `(n, d, C)`; `cumsum(axis=0)` → ilk $i+1$ örneğin sınıf sayımları = sol çocuk. Sağ = toplam − sol.
3. Her pozisyon için ağırlıklı Gini `(n-1, d)`.
4. $X_s[i]=X_s[i+1]$ olan pozisyonlar geçersiz (`np.inf`).
5. Eşitlik kuralı b ile aynı olsun: en küçük skor, sonra küçük özellik, sonra küçük eşik.

- `best_split_np(X, y, C)` → `(j, t, skor)` ya da `None`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def best_split_np(X, y, C):
    """→ (j, t, skor) ya da None. Döngü yok."""
    ...


print(best_split_np(np.array([[1.0], [2], [3], [4], [5], [6]]), np.array([0, 0, 1, 1, 1, 0]), 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Bir **yumuşak karar düğümü** (soft stump):

$$
g(x)=\sigma\!\Big(\frac{w^\top x+b}{\tau}\Big),\qquad
\text{logit}(x)=g(x)\,\ell_{\text{sağ}}+(1-g(x))\,\ell_{\text{sol}},
$$

$\ell_{\text{sol}},\ell_{\text{sağ}}\in\mathbb R^C$ öğrenilen yaprak logit'leri. $\tau\to0$ iken $g$ bir basamak fonksiyonuna, düğüm sert bir eğik bölünmeye dönüşür.

- `SoftStump(nn.Module)` — `__init__(self, d, C, tau=0.5)`, `forward(x)` → `(n, C)` logit.
- `train_soft_stump(X, y, C, steps=500, lr=0.1)` → eğitilmiş `SoftStump`; `CrossEntropyLoss` + `Adam`.

Kontrol: tek bir özelliğin eşiği sınıfı belirlerken (diğer özellikler gürültü) öğrenilen $w$'nin o özelliğe yoğunlaştığını ve doğruluğun ≥ 0.95 olduğunu doğrular.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


class SoftStump(nn.Module):
    def __init__(self, d, C, tau=0.5):
        super().__init__()
        ...

    def forward(self, x):
        ...


def train_soft_stump(X, y, C, steps=500, lr=0.1):
    ...


torch.manual_seed(0)
X = torch.rand(300, 4)
y = (X[:, 2] > 0.6).long()
m = train_soft_stump(X, y, 2)
print((m(X).argmax(1) == y).float().mean())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Boosting (M2.5) her turda örneklere farklı ağırlık verir; bölünme skoru **ağırlıklı** sınıf sayımlarıyla hesaplanır. c'deki cumsum hilesi aynen çalışır: one-hot yerine `one_hot * w[:, None]`.

- `best_split_torch(X, y, C, w=None)` → `(j, t, skor)`; `w` örnek ağırlıkları (`None` ise hepsi 1). **Döngü yok.** Kontrol: tamsayı ağırlıkların satırları kopyalamakla aynı sonucu verdiğini ve c ile eşleştiğini doğrular.

Hız tablosu: $n=2000$, $d=20$ için saf Python `best_split` ile karşılaştırma.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def best_split_torch(X, y, C, w=None):
    """Örnek ağırlıklı Gini. → (j, t, skor) ya da None. Döngü yok."""
    ...


X = torch.tensor([[1.0], [2], [3], [4], [5], [6]])
y = torch.tensor([0, 0, 1, 1, 1, 0])
print(best_split_torch(X, y, 2))
print(best_split_torch(X, y, 2, torch.tensor([1.0, 1, 1, 1, 1, 10])))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Quinlan (1986): karar ağaçlarının tümevarımı (ID3)

[Quinlan — Induction of Decision Trees (Machine Learning 1986)](https://link.springer.com/article/10.1007/BF00116251)

ID3'ü makaledeki gibi bilgi kazancıyla yaz, çok değerli özniteliklere olan yanlılığı (Bölüm 7) bir 'kimlik' özniteliği ekleyerek göster ve kazanç oranıyla (gain ratio) düzelt. Gürültülü veride ağacın aşırı uyumunu çiz; makalenin χ² tabanlı erken durdurmasını ve maliyet-karmaşıklık budamasını (CART) karşılaştır. Makalenin 'Cumartesi sabahı' oyuncak verisinde ağacı birebir üret.


**Adımlar**

1. Sürekli öznitelikler için eşik aramayı (C4.5) ekle.

**Kabul ölçütleri**

- Oyuncak veride makaledeki ağaç.
- Kazanç oranı yanlılığı giderir; budama test hatasını düşürür.

## Kaynaklar

- [ISLP — Bölüm 8.1 The Basics of Decision Trees](https://www.statlearning.com/) — doğrulanmadı
- [Elements of Statistical Learning — §9.2 Tree-Based Methods](https://hastie.su.domains/ElemStatLearn/) — doğrulanmadı
- [scikit-learn User Guide — Decision Trees (CART, Gini)](https://scikit-learn.org/stable/modules/tree.html) — BSD-3
- [Frosst & Hinton — Distilling a Neural Network Into a Soft Decision Tree](https://arxiv.org/abs/1711.09784) — arXiv